In [6]:
## Muat dataset, periksa missing values, tipe data, dan distribusi target.
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset('titanic')

# Pilih kolom yang akan digunakan
cols = [
    'pclass',
    'sex',
    'age',
    'sibsp',
    'parch',
    'fare',
    'embarked',
    'survived'
]

df = df[cols].copy()

# Informasi dataset
print("=== INFORMASI DATASET ===")
print("Shape:", df.shape)
print("Type:\n", df.dtypes)

# Missing values
print("\n=== MISSING VALUES ===")
print(df.isnull().sum())

# Distribusi target
print("\n=== DISTRIBUSI TARGET (SURVIVED) ===")
print(df['survived'].value_counts(normalize=True).round(3))

# Interpretasi
print("\nKeterangan:")
print("survived = 0 (tidak selamat) : ~61,6%")
print("survived = 1 (selamat)       : ~38,4%")

=== INFORMASI DATASET ===
Shape: (891, 8)
Type:
 pclass        int64
sex          object
age         float64
sibsp         int64
parch         int64
fare        float64
embarked     object
survived      int64
dtype: object

=== MISSING VALUES ===
pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
survived      0
dtype: int64

=== DISTRIBUSI TARGET (SURVIVED) ===
survived
0    0.616
1    0.384
Name: proportion, dtype: float64

Keterangan:
survived = 0 (tidak selamat) : ~61,6%
survived = 1 (selamat)       : ~38,4%


In [7]:
## Isi nilai yang hilang sebelum encoding. Gunakan median untuk kolom numerik (robust terhadap outlier) dan modus untuk kolom kategorikal.

# Age: isi nilai yang hilang dengan median
df['age'] = df['age'].fillna(df['age'].median())

# Embarked: isi nilai yang hilang dengan modus
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print('Missing setelah handling:')
print(df.isnull().sum())  # Semua harus 0

Missing setelah handling:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
survived    0
dtype: int64


In [8]:
## Terapkan One-Hot Encoding pada kolom 'sex' dan 'embarked'. Gunakan drop_first=True untuk menghindari dummy variable trap.

# One-Hot Encoding untuk 'sex' dan 'embarked'
df = pd.get_dummies(
    df,
    columns=['sex', 'embarked'],
    drop_first=True,
    dtype=int
)

print('Kolom setelah encoding:')
print(df.columns.tolist())

# Output yang diharapkan:
# ['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']

Kolom setelah encoding:
['pclass', 'age', 'sibsp', 'parch', 'fare', 'survived', 'sex_male', 'embarked_Q', 'embarked_S']


In [9]:
## Bagi data dengan stratifikasi untuk menjaga proporsi kelas 'survived' di train dan test set.
from sklearn.model_selection import train_test_split

X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # proporsi kelas terjaga
)

print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')

print('\nProporsi survived di Train:')
print(y_train.value_counts(normalize=True).round(3))

print('\nProporsi survived di Test:')
print(y_test.value_counts(normalize=True).round(3))

Train: 712 baris
Test : 179 baris

Proporsi survived di Train:
survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Proporsi survived di Test:
survived
0    0.615
1    0.385
Name: proportion, dtype: float64


In [10]:
## Feature Scaling
# Terapkan StandardScaler HANYA pada kolom numerik, fit pada training set, transform pada keduanya.
# Kolom biner hasil OHE tidak perlu di-scale.
from sklearn.preprocessing import StandardScaler

# Hanya kolom numerik yang perlu di-scale
# Kolom biner (sex_male, embarked_Q, embarked_S) tidak perlu
num_cols = ['pclass', 'age', 'sibsp', 'parch', 'fare']

scaler = StandardScaler()

# fit_transform pada training set
# (belajar nilai mean dan standar deviasi dari data training)
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

# transform pada test set
# (menggunakan mean dan standar deviasi dari training set)
X_test[num_cols] = scaler.transform(X_test[num_cols])

print('Mean scaler (dari train):', scaler.mean_.round(2))
print('Std scaler (dari train):', scaler.scale_.round(2))

print()
print('Contoh X_train setelah scaling:')
print(X_train.head(3).round(3))

print('\nData siap dilatih model Machine Learning!')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test : {X_test.shape}, y_test : {y_test.shape}')

Mean scaler (dari train): [ 2.31 29.46  0.49  0.39 31.82]
Std scaler (dari train): [ 0.83 13.03  1.06  0.84 48.03]

Contoh X_train setelah scaling:
     pclass    age  sibsp  parch   fare  sex_male  embarked_Q  embarked_S
692   0.830 -0.112 -0.465 -0.466  0.514         1           0           1
481  -0.371 -0.112 -0.465 -0.466 -0.663         1           0           1
527  -1.571 -0.112 -0.465 -0.466  3.955         1           0           1

Data siap dilatih model Machine Learning!
X_train: (712, 8), y_train: (712,)
X_test : (179, 8), y_test : (179,)
